# Continuous Analog Input Demo (USB6451)

This notebook demonstrates continuous analog input acquisition using the `USB6451` class API.

It can also generate a continuous AO sinewave while reading AI, so you can do a simple loopback measurement
(for example wire `ao0` to `ai0` through safe wiring and limits).

References used:
- `analog_in/cont_voltage_acq_int_clk.py` from `nidaqmx-python-master`
- `analog_out/cont_gen_voltage_wfm_int_clk.py` from `nidaqmx-python-master`
- `USB6451/USB6451 manual.pdf`


## 1. Setup imports


In [ ]:
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


def add_repo_to_path() -> None:
    """Ensure repository root is importable in this notebook session."""
    cwd = Path.cwd()
    if (cwd / "USB6451" / "USB6451.py").exists():
        sys.path.insert(0, str(cwd))
        return

    candidate = cwd.parent.parent
    if (candidate / "USB6451" / "USB6451.py").exists():
        sys.path.insert(0, str(candidate))
        return

    raise RuntimeError("Could not find repository root containing USB6451/USB6451.py")


add_repo_to_path()
from USB6451.USB6451 import USB6451

print("Imports ready.")


## 2. Define acquisition parameters

Set channel list and timing here.


In [ ]:
# Device and channels
# You can use one channel ("ai0") or multiple channels ("ai0", "ai1", ...)
device = "Dev1"
ai_channels = ("ai0",)

# Timing and acquisition controls
sample_rate = 10_000.0          # S/s per channel
samples_per_chunk = 1000        # samples/channel/read
duration_seconds = 2.0          # total acquisition time
read_timeout = 5.0              # per-read timeout

# Input voltage range
min_voltage = -10.0
max_voltage = 10.0

# Optional AO sine generation for loopback measurement
generate_sine_output = True
ao_channel = "ao0"
sine_frequency = 50.0
sine_amplitude = 1.0
sine_offset = 0.0

print("Parameters set.")


## 3. Start optional AO sine + continuous AI acquisition

This cell can start sine generation on AO and then starts AI acquisition.
It reads chunks until the target duration is reached and stores all chunks in memory.


In [ ]:
daq = USB6451()

if generate_sine_output:
    actual_output_frequency = daq.start_continuous_sine_output(
        device=device,
        ao_channel=ao_channel,
        frequency=sine_frequency,
        amplitude=sine_amplitude,
        offset=sine_offset,
        sample_rate=sample_rate,
        min_voltage=min_voltage,
        max_voltage=max_voltage,
        allow_regen=True,
    )
    print("AO sine output started.")
    print(f"Requested AO frequency: {sine_frequency:g} Hz")
    print(f"Actual AO frequency:    {actual_output_frequency:g} Hz")

actual_sample_rate = daq.start_continuous_input(
    device=device,
    ai_channels=ai_channels,
    sample_rate=sample_rate,
    min_voltage=min_voltage,
    max_voltage=max_voltage,
)

print(f"AI started. Requested sample_rate: {sample_rate:g} S/s")
print(f"AI actual sample_rate: {actual_sample_rate:g} S/s")

chunks = []
start_time = time.perf_counter()

while (time.perf_counter() - start_time) < duration_seconds:
    chunk = daq.read_input_chunk(
        samples_per_channel=samples_per_chunk,
        timeout=read_timeout,
    )
    chunks.append(chunk)

daq.stop_input()
if generate_sine_output:
    daq.stop_output()

print("Acquisition stopped.")
print(f"Chunks read: {len(chunks)}")


## 4. Combine chunks and print summary


In [ ]:
if not chunks:
    raise RuntimeError("No data chunks were captured.")

# Each chunk shape: (channels, samples_per_chunk)
data = np.concatenate(chunks, axis=1)
channel_count, total_samples = data.shape

dt = 1.0 / actual_sample_rate
time_axis = np.arange(total_samples) * dt

print(f"Data shape: {data.shape} (channels, samples)")
print(f"Total duration: {time_axis[-1] if total_samples > 1 else 0.0:.6f} s")
print(f"Channel count: {channel_count}")


## 5. Plot acquired waveforms

Readability-first plot: simple time-voltage lines, one trace per channel.


In [ ]:
plt.figure(figsize=(12, 8))

for i in range(channel_count):
    label = ai_channels[i] if i < len(ai_channels) else f"ch{i}"
    plt.plot(time_axis, data[i], label=label)

plt.title("Continuous AI acquisition")
plt.xlabel("Time (s)")
plt.ylabel("Voltage (V)")
plt.grid(True, alpha=0.3)
plt.legend(loc="best")
plt.tight_layout()
plt.show()


## 6. Optional cleanup cell

If you interrupt execution earlier, run this cell to ensure both AI and AO tasks are stopped.


In [ ]:
try:
    daq.stop_input()
except Exception:
    pass

try:
    daq.stop_output()
except Exception:
    pass

try:
    daq.close()
except Exception:
    pass

print("Cleanup complete.")
